To webscrap the audiofiles

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

BASE_URL = "https://whoicf2.whoi.edu/science/B/whalesounds/"
START_URL = "https://whoicf2.whoi.edu/science/B/whalesounds/fullCuts.cfm?SP=BD15F&YR=61"

def fetch_page(url):
    """Fetch HTML and return BeautifulSoup object."""
    resp = requests.get(url)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")

def get_species_links_and_names(start_url, base_url):
    """Return list of (species_name, species_url)."""
    soup = fetch_page(start_url)
    species_data = []
    select_tag = soup.find("select", {"name": "getSpecies"})
    if not select_tag:
        print("[ERROR] Could not find species dropdown.")
        return species_data

    for option in select_tag.find_all("option"):
        value = option.get("value")
        name = option.get_text(strip=True).replace(" ", "_")
        if value and "SP=-1" not in value:  # skip placeholder
            full_url = urljoin(base_url, value)
            species_data.append((name, full_url))
    return species_data

def get_year_links_for_species(species_url, base_url):
    """Get all year links for a given species."""
    soup = fetch_page(species_url)
    year_links = []
    select_tag = soup.find("select", {"name": "pickYear"})
    if not select_tag:
        print("[ERROR] Could not find year dropdown.")
        return year_links

    for option in select_tag.find_all("option"):
        value = option.get("value")
        year_text = option.get_text(strip=True)
        if value and "YR=-1" not in value:  # skip placeholder
            full_url = urljoin(base_url, value)
            year_links.append((year_text, full_url))
    return year_links

def download_wavs_from_page(page_url, save_dir):
    """Download all WAV files from a given year page."""
    soup = fetch_page(page_url)
    download_links = soup.find_all("a", string="Download")
    if not download_links:
        print(f"[WARNING] No 'Download' links on {page_url}")
        return

    os.makedirs(save_dir, exist_ok=True)

    for link in download_links:
        wav_url = urljoin(page_url, link["href"])
        filename = os.path.basename(wav_url)
        save_path = os.path.join(save_dir, filename)

        print(f"[INFO] Downloading: {wav_url}")
        try:
            r = requests.get(wav_url, stream=True)
            r.raise_for_status()
            with open(save_path, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    f.write(chunk)
            print(f"[SUCCESS] Saved: {save_path}")
        except Exception as e:
            print(f"[ERROR] Failed to download {wav_url}: {e}")

if __name__ == "__main__":
    print("[INFO] Getting all species...")
    species_list = get_species_links_and_names(START_URL, BASE_URL)
    print(f"[INFO] Found {len(species_list)} species.")

    for species_name, species_url in species_list:
        print(f"\n[INFO] Processing species: {species_name}")
        year_list = get_year_links_for_species(species_url, BASE_URL)

        # Create a folder for each species only once
        save_dir = os.path.join("downloads", species_name.lower())

        for year_text, year_url in year_list:
            print(f"[INFO]  Year: {year_text}")
            download_wavs_from_page(year_url, save_dir)


[INFO] Getting all species...
[INFO] Found 55 species.

[INFO] Processing species: Atlantic_Spotted_Dolphin
[INFO]  Year: 1961
[INFO] Downloading: https://whoicf2.whoi.edu/science/B/whalesounds/WhaleSounds/61025001.wav
[SUCCESS] Saved: downloads\atlantic_spotted_dolphin\61025001.wav
[INFO] Downloading: https://whoicf2.whoi.edu/science/B/whalesounds/WhaleSounds/61025002.wav
[SUCCESS] Saved: downloads\atlantic_spotted_dolphin\61025002.wav
[INFO] Downloading: https://whoicf2.whoi.edu/science/B/whalesounds/WhaleSounds/61025003.wav
[SUCCESS] Saved: downloads\atlantic_spotted_dolphin\61025003.wav
[INFO] Downloading: https://whoicf2.whoi.edu/science/B/whalesounds/WhaleSounds/61025004.wav
[SUCCESS] Saved: downloads\atlantic_spotted_dolphin\61025004.wav
[INFO] Downloading: https://whoicf2.whoi.edu/science/B/whalesounds/WhaleSounds/61025005.wav
[SUCCESS] Saved: downloads\atlantic_spotted_dolphin\61025005.wav
[INFO] Downloading: https://whoicf2.whoi.edu/science/B/whalesounds/WhaleSounds/61025006.